# 01 — Raw Data Collection | Sentimen Kebijakan Bencana Nasional

Notebook mandiri (tanpa dependensi pada file .py ini). Jalankan cell secara urut.
**Tidak ada scraping otomatis:** `EXECUTE_SCRAPING = False` secara default.
Raw asli disimpan sebagai JSONL; laporan kualitas tidak mengubah atau membuang raw.

In [61]:
# 01 — IMPORT & VERSI
from __future__ import annotations

import hashlib
import json
import os
import re
import time
from collections import Counter
from datetime import datetime, timedelta, timezone
from getpass import getpass
from pathlib import Path
from typing import Any
from zoneinfo import ZoneInfo

import pandas as pd

try:
    from apify_client import ApifyClient
except ImportError as exc:
    raise ImportError(
        'Paket apify-client belum terpasang. Jalankan: python -m pip install apify-client pandas'
    ) from exc

print('Pandas:', pd.__version__)
print('Import berhasil.')

Pandas: 3.0.6
Import berhasil.


In [62]:
# 02 — LOKASI PROJECT & OUTPUT
# Repository memiliki folder bernama "01_Raw Data" (spasi), bukan "01_Raw_Data".
# Jika Jupyter dibuka dari lokasi lain, set PROJECT_ROOT melalui environment variable.
root_env = os.getenv('PROJECT_ROOT')
if root_env:
    PROJECT_ROOT = Path(root_env).expanduser().resolve()
elif (Path.cwd() / '01_Raw Data').is_dir():
    PROJECT_ROOT = Path.cwd().resolve()
elif Path.cwd().name == '01_Raw Data':
    PROJECT_ROOT = Path.cwd().parent.resolve()
else:
    raise RuntimeError(
        'Folder 01_Raw Data tidak ditemukan. Buka notebook dari repository atau set PROJECT_ROOT.'
    )

STAGE_DIR = PROJECT_ROOT / '01_Raw Data'
OUT_DIR = STAGE_DIR / 'outputs'
RAW_DIR = OUT_DIR / 'raw_jsonl'
CSV_DIR = OUT_DIR / 'raw_csv'
AUDIT_DIR = OUT_DIR / 'audit'
STATE_DIR = OUT_DIR / 'state'
REPORT_DIR = OUT_DIR / 'reports'
for folder in [RAW_DIR, CSV_DIR, AUDIT_DIR, STATE_DIR, REPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

CHECKPOINT_FILE = STATE_DIR / 'collection_checkpoint.jsonl'
CONFIG_VERSION = 'policy_v2_2026_10'
ACTOR_ID = 'CJdippxWmn9uRfooo'
WIB = ZoneInfo('Asia/Jakarta')
UTC = timezone.utc

print('Project:', PROJECT_ROOT)
print('Output :', OUT_DIR)
print('Actor  :', ACTOR_ID)

Project: C:\Penelitian Sentimen Bencana Nasional
Output : C:\Penelitian Sentimen Bencana Nasional\01_Raw Data\outputs
Actor  : CJdippxWmn9uRfooo


## Langkah 1 — Konfigurasi kasus, kata kunci, dan query
Ketiga periode **inklusif**. Semua tanggal tweet dan jendela pengambilan dihitung dalam WIB.
Kelompok EVENT harus berpasangan dengan kelompok POLICY. Tidak ada scraping event-only.
Keyword luas (contohnya "status darurat", "pemerintah pusat") ditandai untuk peninjauan,
bukan otomatis dianggap sebagai opini tentang status bencana nasional.

In [81]:
# 03 — KONFIGURASI PENELITIAN (dipertahankan dari notebook semula)
DISASTER_CASES = {
    'lombok_2018': {
        'label': 'Gempa Lombok 2018',
        'start_date': '2018-07-29', 'end_date': '2018-08-31',
        'pilot_date': '2018-08-13',
        'event_terms': [
            'Lombok', 'gempa Lombok', 'gempa bumi Lombok', 'gempa NTB',
            'gempa bumi NTB', 'NTB', 'Nusa Tenggara Barat', 'Mataram',
            'Lombok Utara', 'Lombok Barat', 'Lombok Timur', 'Lombok Tengah'
        ],
        'policy_terms': [
            'bencana nasional', 'status bencana nasional', 'status nasional',
            'penetapan bencana nasional', 'penetapan status bencana',
            'penetapan status', 'status bencana', 'status darurat',
            'bencana daerah', 'status bencana daerah',
            'ditetapkan sebagai bencana nasional', 'ditetapkan bencana nasional',
            'tetapkan bencana nasional', 'menetapkan bencana nasional',
            'jadi bencana nasional', 'jadiin bencana nasional'
        ],
    },
    'palu_2018': {
        'label': 'Gempa dan Tsunami Palu 2018',
        'start_date': '2018-09-28', 'end_date': '2018-10-31',
        'pilot_date': '2018-10-02',
        'event_terms': [
            'Palu', 'gempa Palu', 'gempa bumi Palu', 'tsunami Palu',
            'gempa tsunami Palu', 'Donggala', 'gempa Donggala',
            'tsunami Donggala', 'Sulawesi Tengah', 'gempa Sulawesi Tengah',
            'tsunami Sulawesi Tengah', 'bencana Palu', 'bencana Sulawesi Tengah'
        ],
        'policy_terms': [
            'bencana nasional', 'status bencana nasional', 'status nasional',
            'penetapan bencana nasional', 'penetapan status bencana',
            'penetapan status', 'status bencana', 'status darurat',
            'bencana daerah', 'status bencana daerah',
            'ditetapkan sebagai bencana nasional', 'ditetapkan bencana nasional',
            'tetapkan bencana nasional', 'menetapkan bencana nasional',
            'jadi bencana nasional', 'bantuan internasional'
        ],
    },
    'sumatra_2025': {
        'label': 'Banjir dan Longsor Sumatra 2025',
        'start_date': '2025-11-26', 'end_date': '2025-12-31',
        'pilot_date': '2025-12-03',
        'event_terms': [
            'Aceh', 'banjir Aceh', 'longsor Aceh', 'bencana Aceh',
            'Sumatra Utara', 'Sumatera Utara', 'banjir Sumatra Utara',
            'banjir Sumatera Utara', 'longsor Sumatra Utara',
            'longsor Sumatera Utara', 'bencana Sumatra Utara',
            'bencana Sumatera Utara', 'Sumatra Barat', 'Sumatera Barat',
            'banjir Sumatra Barat', 'banjir Sumatera Barat',
            'longsor Sumatra Barat', 'longsor Sumatera Barat',
            'bencana Sumatra Barat', 'bencana Sumatera Barat',
            'banjir Sumatra', 'banjir Sumatera', 'longsor Sumatra',
            'longsor Sumatera', 'bencana Sumatra', 'bencana Sumatera'
        ],
        'policy_terms': [
            'bencana nasional', 'status bencana nasional', 'status nasional',
            'penetapan bencana nasional', 'penetapan status bencana',
            'penetapan status', 'status bencana', 'status darurat',
            'bencana daerah', 'status bencana daerah',
            'ditetapkan sebagai bencana nasional', 'ditetapkan bencana nasional',
            'tetapkan bencana nasional', 'menetapkan bencana nasional',
            'jadi bencana nasional', 'prioritas nasional',
            'penanganan nasional', 'penanganan berskala nasional',
            'berskala nasional', 'ditangani pemerintah pusat',
            'penanganan pemerintah pusat', 'pemerintah pusat',
            'status daerah', 'status provinsi'
        ],
    },
}

STRONG_POLICY_TERMS = [
    'bencana nasional', 'status bencana nasional',
    'penetapan bencana nasional', 'ditetapkan sebagai bencana nasional',
    'ditetapkan bencana nasional', 'tetapkan bencana nasional',
    'menetapkan bencana nasional', 'jadi bencana nasional',
    'jadiin bencana nasional', 'status nasional'
]

BROAD_POLICY_TERMS = [
    'penetapan status bencana', 'penetapan status', 'status bencana',
    'status darurat', 'bencana daerah', 'status bencana daerah',
    'bantuan internasional', 'prioritas nasional', 'penanganan nasional',
    'penanganan berskala nasional', 'berskala nasional',
    'ditangani pemerintah pusat', 'penanganan pemerintah pusat',
    'pemerintah pusat', 'status daerah', 'status provinsi'
]

# Grup query yang sama seperti notebook lama (5 event × 6 policy / batch).
EVENT_CHUNK_SIZE = 5
POLICY_CHUNK_SIZE = 6


def chunks(items: list[str], size: int):
    if size <= 0:
        raise ValueError('Ukuran batch harus > 0')
    for i in range(0, len(items), size):
        yield items[i:i + size]


def or_query(terms: list[str]) -> str:
    return '(' + ' OR '.join('"' + term.replace('"', '') + '"' for term in terms) + ')'


def make_query_batches(case_config: dict) -> list[dict]:
    batches = []
    for e_index, events in enumerate(chunks(case_config['event_terms'], EVENT_CHUNK_SIZE), 1):
        for p_index, policies in enumerate(chunks(case_config['policy_terms'], POLICY_CHUNK_SIZE), 1):
            # X search: spasi antarkelompok bermakna konjungsi/AND.
            # Keyword kebijakan tetap diwajibkan; hindari literal 'AND' dalam kueri.
            base = f'{or_query(events)} {or_query(policies)}'
            batches.append({
                'query_id': f'E{e_index:02d}_P{p_index:02d}',
                'base_query': base,
                'event_terms': events,
                'policy_terms': policies,
            })
    return batches

for case_id, conf in DISASTER_CASES.items():
    conf['query_batches'] = make_query_batches(conf)
    print(case_id, '| event:', len(conf['event_terms']),
        '| policy:', len(conf['policy_terms']),
        '| query batch:', len(conf['query_batches']))

assert [len(DISASTER_CASES[c]['query_batches']) for c in DISASTER_CASES] == [9, 9, 24]

lombok_2018 | event: 12 | policy: 16 | query batch: 9
palu_2018 | event: 13 | policy: 16 | query batch: 9
sumatra_2025 | event: 26 | policy: 24 | query batch: 24


## Langkah 2 — Query waktu dan pemeriksaan kualitas
`lang:in` dan `since_time`/`until_time` dipilih dari hasil uji Actor sebelumnya.
Filter waktu Actor tidak selalu konsisten; **createdAt selalu divalidasi ulang**.
'PASS' tidak membuktikan data lengkap atau mewakili seluruh opini publik.

In [83]:
# 04 — MODE EKSEKUSI, BATAS BIAYA, & ADAPTIVE
# Semua opsi produksi harus diubah secara sengaja.
# 04 — MODE EKSEKUSI PILOT 8

RUN_MODE = 'pilot'
EXECUTE_SCRAPING = True

MAX_NEW_RUNS = 2
MAX_ITEMS = 20

PILOT_CASES = ('sumatra_2025',)
PILOT_START_HOURS = (6, 9)

PILOT_QUERY_LIMIT = 1
PILOT_HOURS = 3
PILOT_START_HOUR = 6

FULL_WINDOW_HOURS = 24
ENABLE_ADAPTIVE = False
REQUEST_DELAY_SECONDS = 1.0
MAX_ADAPTIVE_DEPTH = 3

assert RUN_MODE in {'pilot', 'full'}
assert 1 <= PILOT_HOURS <= 24 and 0 <= PILOT_START_HOUR <= 23
assert PILOT_START_HOUR + PILOT_HOURS <= 24
assert FULL_WINDOW_HOURS in {1, 6, 12, 24}
assert MAX_NEW_RUNS >= 1 and PILOT_QUERY_LIMIT >= 1
assert set(PILOT_CASES).issubset(DISASTER_CASES)

assert all(
    0 <= hour < 24
    and hour + PILOT_HOURS <= 24
    for hour in PILOT_START_HOURS
)


def wib_dt(day: str, hour: int = 0) -> datetime:
    return datetime.fromisoformat(day).replace(hour=hour, tzinfo=WIB)


def iso_wib(dt: datetime) -> str:
    return dt.astimezone(WIB).strftime('%Y-%m-%d %H:%M:%S%z')


def make_search_query(base_query: str, start_wib: datetime, end_wib: datetime) -> str:
    start_unix = int(start_wib.timestamp())
    end_unix = int(end_wib.timestamp())
    return f'{base_query} lang:in since_time:{start_unix} until_time:{end_unix}'


def task_key(task: dict) -> str:
    identifying = {
        'case': task['case'], 'base_query': task['base_query'],
        'start_wib': iso_wib(task['start_wib']),
        'end_wib': iso_wib(task['end_wib']),
        'config_version': CONFIG_VERSION, 'actor_id': ACTOR_ID,
    }
    digest = hashlib.sha256(
        json.dumps(identifying, sort_keys=True, ensure_ascii=False).encode('utf-8')
    ).hexdigest()[:20]
    return f"{task['case']}_{digest}"


def task_info(case_id: str, batch: dict, start: datetime, end: datetime, depth=0) -> dict:
    task = {
        'case': case_id, 'query_id': batch['query_id'],
        'base_query': batch['base_query'],
        'start_wib': start, 'end_wib': end, 'depth': depth,
    }
    task['task_id'] = task_key(task)
    task['query'] = make_search_query(task['base_query'], start, end)
    return task


def date_windows(conf: dict, hours: int):
    start = wib_dt(conf['start_date'])
    end_exclusive = wib_dt(conf['end_date']) + timedelta(days=1)
    step = timedelta(hours=hours)
    while start < end_exclusive:
        stop = min(start + step, end_exclusive)
        yield start, stop
        start = stop


def build_tasks(mode: str = RUN_MODE) -> list[dict]:
    # Round-robin antarkasus, agar batas MAX_NEW_RUNS tidak hanya
    # menghabiskan kuota untuk Lombok ketika full mode dimulai.
    case_tasks = {}
    for case_id, conf in DISASTER_CASES.items():
        group = []
        # if mode == 'pilot':
        #     start = wib_dt(conf['pilot_date'], PILOT_START_HOUR)
        #     windows = [(start, start + timedelta(hours=PILOT_HOURS))]
        #     batches = conf['query_batches'][:PILOT_QUERY_LIMIT]
        if mode == 'pilot':
            # Hanya proses kasus yang dipilih
            if case_id not in PILOT_CASES:
                case_tasks[case_id] = []
                continue

            # Buat window khusus Pilot 8
            windows = []

            for hour in PILOT_START_HOURS:

                start = wib_dt(
                    conf['pilot_date'],
                    hour
                )

                end = start + timedelta(
                    hours=PILOT_HOURS
                )

                windows.append((start, end))
            # Gunakan query utama saja
            batches = conf['query_batches'][:PILOT_QUERY_LIMIT]
            
        elif mode == 'full':
            windows = list(date_windows(conf, FULL_WINDOW_HOURS))
            batches = conf['query_batches']
        else:
            raise ValueError('mode harus pilot atau full')
        for start, end in windows:
            for batch in batches:
                group.append(task_info(case_id, batch, start, end))
        case_tasks[case_id] = group

    tasks = []
    longest = max(len(group) for group in case_tasks.values())
    for index in range(longest):
        for case_id in DISASTER_CASES:
            if index < len(case_tasks[case_id]):
                tasks.append(case_tasks[case_id][index])
    return tasks


def preview_plan(mode=RUN_MODE):
    tasks = build_tasks(mode)
    print('=' * 65)
    print('PREVIEW (tidak memanggil Apify) | Mode:', mode)
    print('Config:', CONFIG_VERSION, '| Requests awal:', len(tasks))
    print('Max run baru pada satu eksekusi:', MAX_NEW_RUNS)
    for case in DISASTER_CASES:
        n = sum(t['case'] == case for t in tasks)
        print(f'  {case}: {n} task')
    print('Perkiraan biaya uang tidak dapat dihitung hanya dari jumlah task.')
    for task in tasks[:min(3, len(tasks))]:
        print('\n-', task['case'], task['query_id'])
        print(' ', iso_wib(task['start_wib']), 's/d', iso_wib(task['end_wib']))
        print(' ', task['query'])
    return tasks

TASK_PREVIEW = preview_plan()

# VALIDASI FINAL PILOT 8 — TANPA API CALL

expected = [
    ("2025-12-03 06:00", "2025-12-03 09:00"),
    ("2025-12-03 09:00", "2025-12-03 12:00"),
]

actual = [
    (
        task["start_wib"].strftime("%Y-%m-%d %H:%M"),
        task["end_wib"].strftime("%Y-%m-%d %H:%M"),
    )
    for task in TASK_PREVIEW
]

assert len(TASK_PREVIEW) == 2
assert all(t["case"] == "sumatra_2025" for t in TASK_PREVIEW)
assert all(t["query_id"] == "E01_P01" for t in TASK_PREVIEW)
assert actual == expected, (
    f"Window Pilot 8 tidak sesuai: {actual}"
)

print("PILOT 8 PREFLIGHT: PASS")

PREVIEW (tidak memanggil Apify) | Mode: pilot
Config: policy_v2_2026_10 | Requests awal: 2
Max run baru pada satu eksekusi: 2
  lombok_2018: 0 task
  palu_2018: 0 task
  sumatra_2025: 2 task
Perkiraan biaya uang tidak dapat dihitung hanya dari jumlah task.

- sumatra_2025 E01_P01
  2025-12-03 06:00:00+0700 s/d 2025-12-03 09:00:00+0700
  ("Aceh" OR "banjir Aceh" OR "longsor Aceh" OR "bencana Aceh" OR "Sumatra Utara") ("bencana nasional" OR "status bencana nasional" OR "status nasional" OR "penetapan bencana nasional" OR "penetapan status bencana" OR "penetapan status") lang:in since_time:1764716400 until_time:1764727200

- sumatra_2025 E01_P01
  2025-12-03 09:00:00+0700 s/d 2025-12-03 12:00:00+0700
  ("Aceh" OR "banjir Aceh" OR "longsor Aceh" OR "bencana Aceh" OR "Sumatra Utara") ("bencana nasional" OR "status bencana nasional" OR "status nasional" OR "penetapan bencana nasional" OR "penetapan status bencana" OR "penetapan status") lang:in since_time:1764727200 until_time:1764738000
PIL

In [77]:
# 05 — CEK OPERATOR & METADATA SEBELUM PANGGILAN

def preflight(tasks: list[dict]):
    assert len(DISASTER_CASES) == 3
    assert {c: len(v['query_batches']) for c, v in DISASTER_CASES.items()} == {
        'lombok_2018': 9, 'palu_2018': 9, 'sumatra_2025': 24
    }
    assert MAX_NEW_RUNS > 0
    for task in tasks:
        assert task['start_wib'] < task['end_wib']
        assert 'lang:in' in task['query']
        assert 'since_time:' in task['query'] and 'until_time:' in task['query']
        assert ' OR ' in task['base_query']
        assert ') (' in task['base_query'], 'Query harus EVENT dan POLICY'
        assert ' AND ' not in task['base_query'], 'Hapus literal AND dari query'
        assert task['case'] in DISASTER_CASES
    assert len({t['task_id'] for t in tasks}) == len(tasks)
    return True

print('Preflight:', 'PASS' if preflight(TASK_PREVIEW) else 'FAIL')

Preflight: PASS


## Langkah 3 — Validasi Actor, format tweet, dan pembagian jendela
Yang divalidasi: tweet asli (`type=tweet`), ID, `createdAt` parseable, tanggal dalam jendela,
serta dua kelompok keyword. Hasil yang meragukan tetap disimpan ke arsip raw dan audit.
Catatan: pencocokan teks adalah indikator otomatis, bukan keputusan relevansi semantik final.

In [78]:
# 06 — FUNGSI AUDIT DATA (TANPA API CALL)

def match_any_phrase(text: str, terms: list[str]) -> bool:
    # Hindari pencocokan sebagian kata (misalnya 'Palu' di 'kepaluan').
    sample = str(text).casefold()
    for term in terms:
        phrase = term.casefold()
        pat = r'(?<!\w)' + re.escape(phrase) + r'(?!\w)'
        if re.search(pat, sample):
            return True
    return False


def parse_twitter_timestamp(values: pd.Series) -> pd.Series:
    result = pd.to_datetime(
        values, format='%a %b %d %H:%M:%S %z %Y',
        errors='coerce', utc=True
    )
    # Cadangan apabila format sumber suatu hari berubah.
    missing = result.isna() & values.notna()
    if missing.any():
        result.loc[missing] = pd.to_datetime(
            values.loc[missing], errors='coerce', utc=True
        )
    return result


def audit_items(items: list[dict], task: dict) -> tuple[pd.DataFrame, dict]:
    df = pd.DataFrame(items)
    initial_columns = list(df.columns)
    for column in ['type', 'id', 'createdAt', 'text', 'lang']:
        if column not in df.columns:
            df[column] = pd.Series([None] * len(df), dtype='object')

    timestamps = parse_twitter_timestamp(df['createdAt'])
    start_utc = pd.Timestamp(task['start_wib'].astimezone(UTC))
    end_utc = pd.Timestamp(task['end_wib'].astimezone(UTC))

    df['audit_created_at_utc'] = timestamps
    df['audit_is_real_tweet'] = df['type'].eq('tweet')
    df['audit_in_window'] = timestamps.ge(start_utc) & timestamps.lt(end_utc)

    # Audit menggunakan seluruh daftar istilah yang relevan dengan kasus.
    # Term yang ada dalam batch query sendiri juga wajib dicek melalui hasil.
    event_terms = DISASTER_CASES[task['case']]['event_terms']
    policy_terms = DISASTER_CASES[task['case']]['policy_terms']
    texts = df['text'].fillna('').astype(str)
    df['audit_has_event'] = texts.map(lambda x: match_any_phrase(x, event_terms))
    df['audit_has_policy'] = texts.map(lambda x: match_any_phrase(x, policy_terms))
    df['audit_policy_strong'] = texts.map(lambda x: match_any_phrase(x, STRONG_POLICY_TERMS))
    df['audit_candidate'] = (
        df['audit_is_real_tweet'] & df['audit_in_window']
        & df['audit_has_event'] & df['audit_has_policy']
    )

    valid_ids = df.loc[df['audit_is_real_tweet'], 'id'].dropna().astype(str)
    n_mock = int(df['type'].eq('mock_tweet').sum())
    n_real = int(df['audit_is_real_tweet'].sum())
    n_in = int((df['audit_is_real_tweet'] & df['audit_in_window']).sum())
    n_candidate = int(df['audit_candidate'].sum())

    metrics = {
        'rows_raw': len(df),
        'rows_real_tweet': n_real,
        'rows_mock': n_mock,
        'rows_unknown_type': int((~df['type'].isin(['tweet', 'mock_tweet'])).sum()),
        'rows_real_in_window': n_in,
        'rows_real_out_window': int((df['audit_is_real_tweet'] & ~df['audit_in_window']).sum()),
        'rows_real_missing_timestamp': int((df['audit_is_real_tweet'] & timestamps.isna()).sum()),
        'rows_event_match': int((df['audit_is_real_tweet'] & df['audit_has_event']).sum()),
        'rows_policy_match': int((df['audit_is_real_tweet'] & df['audit_has_policy']).sum()),
        'rows_event_and_policy_in_window': n_candidate,
        'rows_strong_policy_candidate': int((df['audit_candidate'] & df['audit_policy_strong']).sum()),
        'unique_real_ids': int(valid_ids.nunique()),
        'duplicate_real_ids_in_dataset': int(valid_ids.duplicated().sum()),
        'raw_columns': initial_columns,
        'created_min_utc': timestamps.min().isoformat() if timestamps.notna().any() else None,
        'created_max_utc': timestamps.max().isoformat() if timestamps.notna().any() else None,
    }
    return df, metrics


def quality_flags(metrics: dict) -> list[str]:
    flags = []
    if metrics['rows_raw'] == 0:
        flags.append('EMPTY_SEARCH')
    if metrics['rows_mock']:
        flags.append('MOCK_PRESENT')
    if metrics['rows_raw'] > 0 and metrics['rows_real_tweet'] == 0:
        flags.append('NO_REAL_TWEET')
    if metrics['rows_real_out_window']:
        flags.append('OUT_OF_WINDOW')
    if metrics['rows_real_missing_timestamp']:
        flags.append('BAD_TIMESTAMP')
    if metrics['rows_real_tweet'] and metrics['rows_event_and_policy_in_window'] == 0:
        flags.append('NO_POLICY_CANDIDATE')
    if metrics['duplicate_real_ids_in_dataset'] > 0:
        flags.append('DUPLICATE_IDS')
    # MAX_ITEMS bukan hard-limit aktual Actor. Ini indikator risiko saja.
    if metrics['unique_real_ids'] >= MAX_ITEMS:
        flags.append('POSSIBLE_TRUNCATION')
    return flags or ['BASIC_CHECKS_PASS']

# ============================================================
# UNIT TEST DINAMIS - AUDIT DATA
# Tidak memanggil Apify
# ============================================================

# Ambil task pertama dari konfigurasi pilot terbaru
_t = TASK_PREVIEW[0]

# Buat timestamp yang selalu berada dalam window pilot
test_datetime = (
    _t["start_wib"] + timedelta(hours=1)
).astimezone(UTC)

test_created_at = test_datetime.strftime(
    "%a %b %d %H:%M:%S %z %Y"
)

# Buat data dummy untuk pengujian
_sample = [
    {
        "type": "mock_tweet",
        "id": "demo-mock",
        "text": "dummy"
    },
    {
        "type": "tweet",
        "id": "demo-real",
        "text": (
            f"Haruskah bencana "
            f"{DISASTER_CASES[_t['case']]['event_terms'][0]} "
            f"ditetapkan sebagai bencana nasional?"
        ),
        "createdAt": test_created_at,
        "lang": "in"
    }
]

# Jalankan audit
_audit_demo, _m_demo = audit_items(
    _sample,
    _t
)

print("=" * 65)
print("UNIT TEST AUDIT DATA")
print("=" * 65)

print("Kasus            :", _t["case"])
print("Window mulai     :", _t["start_wib"])
print("Window akhir     :", _t["end_wib"])
print("Timestamp dummy  :", test_created_at)

print("\nHASIL VALIDASI:")
print("Mock tweet       :", _m_demo["rows_mock"])
print(
    "Kandidat valid   :",
    _m_demo["rows_event_and_policy_in_window"]
)

# Pemeriksaan otomatis
assert _m_demo["rows_mock"] == 1, (
    "Jumlah mock tweet tidak sesuai."
)

assert _m_demo["rows_event_and_policy_in_window"] == 1, (
    "Tweet dummy gagal lolos validasi."
)

print("\nUNIT TEST: PASS")

UNIT TEST AUDIT DATA
Kasus            : sumatra_2025
Window mulai     : 2025-12-03 06:00:00+07:00
Window akhir     : 2025-12-03 12:00:00+07:00
Timestamp dummy  : Wed Dec 03 00:00:00 +0000 2025

HASIL VALIDASI:
Mock tweet       : 1
Kandidat valid   : 1

UNIT TEST: PASS


## Langkah 4 — Penyimpanan raw dan checkpoint
Satu folder per kasus. JSONL mempertahankan objek tweet original, audit CSV adalah hasil turunan.
Checkpoint memuat hash task (mempertimbangkan query dan waktu), run ID, dataset ID dan indikator kualitas.
Task yang sudah tersimpan tidak dipanggil ulang saat notebook dijalankan ulang.

In [67]:
# 07 — HELPER CHECKPOINT DAN PENYIMPANAN

def json_write_atomic(path: Path, items: list[dict]):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    with tmp.open('w', encoding='utf-8') as f:
        for item in items:
            f.write(json.dumps(item, ensure_ascii=False, default=str) + '\n')
    tmp.replace(path)


def csv_write_atomic(path: Path, df: pd.DataFrame):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    df.to_csv(tmp, index=False, encoding='utf-8-sig')
    tmp.replace(path)


def append_checkpoint(record: dict):
    CHECKPOINT_FILE.parent.mkdir(parents=True, exist_ok=True)
    with CHECKPOINT_FILE.open('a', encoding='utf-8') as f:
        f.write(json.dumps(record, ensure_ascii=False, default=str) + '\n')


def load_checkpoint() -> dict[str, dict]:
    newest = {}
    if not CHECKPOINT_FILE.exists():
        return newest
    with CHECKPOINT_FILE.open(encoding='utf-8') as f:
        for line in f:
            if not line.strip():
                continue
            try:
                entry = json.loads(line)
            except json.JSONDecodeError as exc:
                raise RuntimeError('Checkpoint rusak. Periksa file sebelum lanjut.') from exc
            newest[entry['task_id']] = entry
    return newest


def make_run_record(task: dict, run_dict: dict, metrics: dict,
                    flags: list[str], raw_path: Path, audit_path: Path) -> dict:
    return {
        'task_id': task['task_id'], 'config_version': CONFIG_VERSION,
        'status': 'SAVED', 'case': task['case'], 'query_id': task['query_id'],
        'depth': task['depth'], 'actor_id': ACTOR_ID,
        'run_id': run_dict.get('id'),
        'dataset_id': run_dict.get('defaultDatasetId') or run_dict.get('default_dataset_id'),
        'actor_status': run_dict.get('status'),
        'query': task['query'], 'base_query': task['base_query'],
        'start_wib': iso_wib(task['start_wib']),
        'end_wib': iso_wib(task['end_wib']),
        'start_utc': task['start_wib'].astimezone(UTC).isoformat(),
        'end_utc': task['end_wib'].astimezone(UTC).isoformat(),
        'quality_flags': flags, **metrics,
        'raw_path': str(raw_path.relative_to(PROJECT_ROOT)),
        'audit_path': str(audit_path.relative_to(PROJECT_ROOT)),
        'saved_at_utc': datetime.now(UTC).isoformat(),
    }


def run_to_dict(run: Any) -> dict:
    if isinstance(run, dict):
        return run
    if hasattr(run, 'model_dump'):
        return run.model_dump(by_alias=True)
    if hasattr(run, 'to_dict'):
        return run.to_dict()
    if hasattr(run, 'default_dataset_id'):
        return {
            'default_dataset_id': run.default_dataset_id,
            'id': getattr(run, 'id', None),
            'status': getattr(run, 'status', None),
        }
    raise TypeError(f'Tipe Apify Run tidak dikenal: {type(run)}')


def get_apify_client() -> ApifyClient:
    token = os.getenv('APIFY_API_TOKEN', '').strip()
    if not token:
        token = getpass('Apify API token (tidak tampil): ').strip()
    if not token:
        raise RuntimeError('API token kosong. Tidak dapat menjalankan Actor.')
    # Jangan cetak atau simpan token dalam notebook, checkpoint, atau GitHub.
    return ApifyClient(token)

In [68]:
# CEK BIAYA RUN APIFY YANG SUDAH SELESAI
# Tidak menjalankan scraping baru

client = get_apify_client()
cost_records = []

for record in load_checkpoint().values():
    run_id = record.get("run_id")

    if not run_id:
        continue

    run = client.run(run_id).get()
    run_info = run_to_dict(run)

    cost_usd = run_info.get(
        "usageTotalUsd",
        run_info.get("usage_total_usd")
    )

    cost_records.append({
        "case": record.get("case"),
        "run_id": run_id,
        "rows_raw": record.get("rows_raw", 0),
        "unique_ids": record.get("unique_real_ids", 0),
        "mock": record.get("rows_mock", 0),
        "cost_usd": cost_usd
    })

df_cost = pd.DataFrame(cost_records)

display(df_cost)

print(
    "Total biaya pilot yang tercatat: $",
    round(df_cost["cost_usd"].sum(), 4)
)

,case,run_id,rows_raw,unique_ids,mock,cost_usd
0,lombok_2018,WKEehIa11ega6QAns,27,12,3,0.00675
1,palu_2018,g2TzJZ81z6RXD3BDe,27,12,3,0.00675
2,sumatra_2025,wFTtrejqJQjETXizC,40,20,0,0.01000
3,lombok_2018,T9gFSdbhCNBC4Nuhi,19,4,11,0.00475
4,palu_2018,Lq3LaSifbNklxrXch,15,4,11,0.00375
5,sumatra_2025,NObgVrjlsxgFc7R8u,25,10,5,0.00625
6,lombok_2018,UeD4R5nIRMVKrkScz,40,20,0,0.01000
7,palu_2018,ekeFKMJe6Egd5jYtn,40,20,0,0.01000
8,sumatra_2025,0r1K4NF8keySrowap,40,20,0,0.01000
9,lombok_2018,aqgBqV2YQZaX72P6p,15,0,15,0.00375


Total biaya pilot yang tercatat: $ 0.1245


In [69]:
# 08 — ADAPTIVE SPLIT, DENGAN BUDGET LIMIT
# Split bukan bukti kelengkapan; hasil puncak selalu perlu ditinjau.
# ENABLE_ADAPTIVE=False secara default demi keselamatan biaya.

def split_task(task: dict) -> list[dict]:
    if task['depth'] >= MAX_ADAPTIVE_DEPTH:
        return []
    total_seconds = (task['end_wib'] - task['start_wib']).total_seconds()
    if total_seconds > 6 * 3600:
        step = timedelta(hours=6)
    elif total_seconds > 1 * 3600:
        step = timedelta(hours=1)
    elif total_seconds > 15 * 60:
        step = timedelta(minutes=15)
    else:
        return []

    result = []
    start = task['start_wib']
    while start < task['end_wib']:
        stop = min(start + step, task['end_wib'])
        # Children tetap memiliki query event+policy yang sama.
        info = {
            'query_id': task['query_id'],
            'base_query': task['base_query'],
        }
        result.append(task_info(task['case'], info, start, stop,
                                depth=task['depth'] + 1))
        start = stop
    return result

## Langkah 5 — Pilot → validasi → full collection secara terkendali
**Wajib periksa output preview sebelum mengubah EXECUTE_SCRAPING.**
`MAX_NEW_RUNS` menjadi limit keras banyaknya `client.actor(...).call()` per eksekusi.
Jangan anggap jendela selesai sepenuhnya bila ada `POSSIBLE_TRUNCATION`.

In [70]:
# 09 — WORKER (MENERIMA CLIENT, MUDAH DITEST TANPA API)

def collect_one(client, task: dict) -> dict:
    run_input = {
        'searchTerms': [task['query']],
        'maxItems': MAX_ITEMS,
        'queryType': 'Latest',
    }
    started = time.monotonic()
    run_dict = run_to_dict(client.actor(ACTOR_ID).call(run_input=run_input))
    actor_status = run_dict.get('status')
    if actor_status != 'SUCCEEDED':
        raise RuntimeError(f'Actor status: {actor_status}')

    dataset_id = run_dict.get('defaultDatasetId') or run_dict.get('default_dataset_id')
    if not dataset_id:
        raise ValueError('defaultDatasetId tidak ada dalam Run.')
    items = list(client.dataset(dataset_id).iterate_items())
    if not all(isinstance(x, dict) for x in items):
        raise TypeError('Actor mengembalikan item bukan dictionary.')

    # Simpan raw SEBELUM audit, tetap termasuk mock atau item di luar tanggal.
    raw_file = RAW_DIR / task['case'] / f"{task['task_id']}.jsonl"
    json_write_atomic(raw_file, items)
    if items:
        csv_write_atomic(CSV_DIR / task['case'] / f"{task['task_id']}.csv", pd.DataFrame(items))

    df_audit, metrics = audit_items(items, task)
    flags = quality_flags(metrics)
    audit_file = AUDIT_DIR / task['case'] / f"{task['task_id']}_audit.csv"
    csv_write_atomic(audit_file, df_audit)

    record = make_run_record(task, run_dict, metrics, flags, raw_file, audit_file)
    record['elapsed_seconds'] = round(time.monotonic() - started, 2)
    append_checkpoint(record)
    return record


def run_collection(client, mode: str = RUN_MODE,
                max_new_runs: int = MAX_NEW_RUNS,
                enable_adaptive: bool = ENABLE_ADAPTIVE):
    tasks = build_tasks(mode)
    preflight(tasks)
    state = load_checkpoint()
    used = 0
    failed_in_this_run = set()
    succeeded = []
    errors = []

    def visit(task):
        nonlocal used
        if used >= max_new_runs:
            return

        existing = state.get(task['task_id'])
        if existing and existing.get('status') == 'SAVED':
            record = existing
        else:
            if task['task_id'] in failed_in_this_run:
                return
            # Hard budget guard: increment SEBELUM call.
            used += 1
            print(f"\n[{used}/{max_new_runs}] {task['case']} {task['query_id']}"
                f" | {iso_wib(task['start_wib'])} → {iso_wib(task['end_wib'])}")
            try:
                record = collect_one(client, task)
                state[task['task_id']] = record
                succeeded.append(record)
                print('  Raw:', record['rows_raw'],
                    '| In-window:', record['rows_real_in_window'],
                    '| Policy candidates:', record['rows_event_and_policy_in_window'])
                print('  Flags:', ', '.join(record['quality_flags']))
            except Exception as exc:
                # Catatan error untuk audit; retry hanya pada eksekusi berikutnya.
                failed_in_this_run.add(task['task_id'])
                entry = {
                    'task_id': task['task_id'], 'status': 'FAILED',
                    'case': task['case'], 'query': task['query'],
                    'timestamp_utc': datetime.now(UTC).isoformat(),
                    'error': f'{type(exc).__name__}: {exc}',
                }
                append_checkpoint(entry)
                state[task['task_id']] = entry
                errors.append(entry)
                print('  ERROR:', entry['error'])
                return
            finally:
                if REQUEST_DELAY_SECONDS > 0 and used < max_new_runs:
                    time.sleep(REQUEST_DELAY_SECONDS)

        # Adaptive hanya setelah raw parent tersimpan; child ditambahkan
        # bertahap dan masih tunduk pada MAX_NEW_RUNS.
        if (
            enable_adaptive
            and 'POSSIBLE_TRUNCATION' in record.get('quality_flags', [])
        ):
            for child in split_task(task):
                if used >= max_new_runs:
                    break
                visit(child)

    for task in tasks:
        if used >= max_new_runs:
            break
        visit(task)

    print('\nSELESAI: Actor calls baru:', used,
        '| Saved baru:', len(succeeded),
        '| Failed:', len(errors))
    return succeeded, errors

In [84]:
# 10 — GERBANG EKSEKUSI: TIDAK MENJALANKAN SCRAPING SECARA OTOMATIS
print('\nGerbang eksekusi')
print('RUN_MODE:', RUN_MODE, '| EXECUTE_SCRAPING:', EXECUTE_SCRAPING)
print('MAX_NEW_RUNS:', MAX_NEW_RUNS, '| ENABLE_ADAPTIVE:', ENABLE_ADAPTIVE)

if EXECUTE_SCRAPING:
    # Hanya meminta token ketika scraping benar-benar dimulai.
    _client = get_apify_client()
    _saved, _errors = run_collection(
        _client, mode=RUN_MODE, max_new_runs=MAX_NEW_RUNS,
        enable_adaptive=ENABLE_ADAPTIVE
    )
else:
    print('PREVIEW ONLY — tidak ada pemanggilan berbayar ke Apify.')


Gerbang eksekusi
RUN_MODE: pilot | EXECUTE_SCRAPING: True
MAX_NEW_RUNS: 2 | ENABLE_ADAPTIVE: False

[1/2] sumatra_2025 E01_P01 | 2025-12-03 06:00:00+0700 → 2025-12-03 09:00:00+0700


[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:Msm9XEAwVZY5rurtX] -> Status: RUNNING, Message: 
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:Msm9XEAwVZY5rurtX] -> 2026-10-10T15:32:18.725Z ACTOR: Pulling container image of build 0n0YJ6lOK1MKTzsdX from registry.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:Msm9XEAwVZY5rurtX] -> 2026-10-10T15:32:18.757Z ACTOR: Creating container.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:Msm9XEAwVZY5rurtX] -> 2026-10-10T15:32:18.815Z ACTOR: Starting container.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:Msm9XEAwVZY5rurtX] -> 2026-10-10T15:32:18.816Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:Msm9XEAwVZY5rurtX] -> 2026-10-10T15:32:24.670Z Actor is running on the Apify platform, `disable_browser_sandbox` was changed to True.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest ru

  Raw: 20 | In-window: 10 | Policy candidates: 10
  Flags: MOCK_PRESENT, DUPLICATE_IDS

[2/2] sumatra_2025 E01_P01 | 2025-12-03 09:00:00+0700 → 2025-12-03 12:00:00+0700


[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:OE1fh3nYzgeJuIAbL] -> Status: RUNNING, Message: 
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:OE1fh3nYzgeJuIAbL] -> 2026-10-10T15:32:47.564Z ACTOR: Pulling container image of build 0n0YJ6lOK1MKTzsdX from registry.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:OE1fh3nYzgeJuIAbL] -> 2026-10-10T15:32:47.566Z ACTOR: Creating container.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:OE1fh3nYzgeJuIAbL] -> 2026-10-10T15:32:47.638Z ACTOR: Starting container.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:OE1fh3nYzgeJuIAbL] -> 2026-10-10T15:32:47.638Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:OE1fh3nYzgeJuIAbL] -> 2026-10-10T15:32:54.383Z Actor is running on the Apify platform, `disable_browser_sandbox` was changed to True.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest ru

  Raw: 25 | In-window: 20 | Policy candidates: 18
  Flags: MOCK_PRESENT, DUPLICATE_IDS

SELESAI: Actor calls baru: 2 | Saved baru: 2 | Failed: 0


In [72]:
# 11 — LAPORAN PROGRES DAN QUALITY (bisa dijalankan tanpa scraping)

def make_collection_report():
    state = load_checkpoint()
    if not state:
        print('Belum ada checkpoint. Jalankan pilot bila siap.')
        return pd.DataFrame()
    report = pd.DataFrame(state.values())
    if 'quality_flags' in report.columns:
        report['quality_flags'] = report['quality_flags'].apply(
            lambda val: ', '.join(val) if isinstance(val, list) else str(val)
        )
    columns = [
        'case', 'query_id', 'status', 'rows_raw', 'rows_mock',
        'rows_real_in_window', 'rows_event_and_policy_in_window',
        'quality_flags', 'query', 'dataset_id', 'run_id', 'raw_path'
    ]
    for col in columns:
        if col not in report.columns:
            report[col] = None
    report = report[columns].sort_values(['case', 'query_id'], na_position='last')
    path = REPORT_DIR / 'collection_summary.csv'
    csv_write_atomic(path, report)
    print('Laporan:', path)
    print(report[['case', 'query_id', 'status', 'rows_raw',
                'rows_real_in_window', 'rows_event_and_policy_in_window',
                'quality_flags']].to_string(index=False, max_rows=30))
    return report

REPORT = make_collection_report()

Laporan: C:\Penelitian Sentimen Bencana Nasional\01_Raw Data\outputs\reports\collection_summary.csv
        case query_id status  rows_raw  rows_real_in_window  rows_event_and_policy_in_window                      quality_flags
 lombok_2018  E01_P01  SAVED        27                   24                               24  MOCK_PRESENT, POSSIBLE_TRUNCATION
 lombok_2018  E01_P01  SAVED        40                   40                               40 DUPLICATE_IDS, POSSIBLE_TRUNCATION
 lombok_2018  E01_P01  SAVED        15                    0                                0        MOCK_PRESENT, NO_REAL_TWEET
 lombok_2018  E01_P01  SAVED        26                   22                               20        MOCK_PRESENT, DUPLICATE_IDS
 lombok_2018  E01_P01  SAVED        19                    8                                8        MOCK_PRESENT, DUPLICATE_IDS
 lombok_2018  E01_P01  SAVED        15                   15                               15                  BASIC_CHECKS_PASS
 lom

In [73]:
import pandas as pd

records = [
    r for r in load_checkpoint().values()
    if r.get("status") == "SAVED"
]

if records:
    df_quality = pd.DataFrame(records)

    kolom = [
        "case",
        "query_id",
        "rows_raw",
        "rows_mock",
        "rows_real_in_window",
        "unique_real_ids",
        "duplicate_real_ids_in_dataset",
        "rows_event_and_policy_in_window"
    ]

    display(df_quality[kolom])

    print("\nTOTAL METRIK PER KASUS:")
    display(
        df_quality.groupby("case")[
            [
                "rows_raw",
                "rows_mock",
                "unique_real_ids",
                "duplicate_real_ids_in_dataset"
            ]
        ].sum()
    )
else:
    print("Belum ada hasil tersimpan.")

,case,query_id,rows_raw,rows_mock,rows_real_in_window,unique_real_ids,duplicate_real_ids_in_dataset,rows_event_and_policy_in_window
0,lombok_2018,E01_P01,27,3,24,12,12,24
1,palu_2018,E01_P01,27,3,24,12,12,24
2,sumatra_2025,E01_P01,40,0,40,20,20,38
3,lombok_2018,E01_P02,19,11,8,4,4,8
4,palu_2018,E01_P02,15,11,4,4,0,4
5,sumatra_2025,E01_P02,25,5,20,10,10,18
6,lombok_2018,E01_P01,40,0,40,20,20,40
7,palu_2018,E01_P01,40,0,40,20,20,36
8,sumatra_2025,E01_P01,40,0,40,20,20,40
9,lombok_2018,E01_P01,15,15,0,0,0,0



TOTAL METRIK PER KASUS:


,rows_raw,rows_mock,unique_real_ids,duplicate_real_ids_in_dataset
case,,,,
lombok_2018,161,44,66,51
palu_2018,181,39,73,69
sumatra_2025,222,23,110,89


In [74]:
# 12 — CONTOH TWEET KANDIDAT UNTUK TINJAUAN MANUAL (TANPA SCRAPING)
def preview_policy_samples(max_each_case=5):
    if max_each_case <= 0:
        raise ValueError('max_each_case harus > 0')
    state = load_checkpoint()
    if not state:
        print('Belum ada hasil scraping tersimpan untuk ditinjau.')
        return {}

    examples = {}
    for case_id in DISASTER_CASES:
        found = []
        for record in state.values():
            if record.get('status') != 'SAVED' or record.get('case') != case_id:
                continue
            audit_path_str = record.get('audit_path')
            if not audit_path_str:
                continue
            path = PROJECT_ROOT / audit_path_str
            if not path.is_file():
                continue
            df = pd.read_csv(path, dtype={'id': 'string'})
            if df.empty or 'audit_candidate' not in df.columns:
                continue
            good = df[
                df['audit_candidate'].astype(str).str.casefold().eq('true')
            ]
            for _, row in good.iterrows():
                found.append({
                    'id': str(row.get('id', '')),
                    'createdAt': str(row.get('createdAt', '')),
                    'text': str(row.get('text', '')),
                })
                if len(found) >= max_each_case:
                    break
            if len(found) >= max_each_case:
                break
        examples[case_id] = found
        print('\n' + '=' * 65)
        print(case_id, '| contoh kandidat otomatis:', len(found))
        for i, item in enumerate(found, 1):
            print(f"{i}. [{item['createdAt']}] {item['text'][:300]}")
        if not found:
            print('Belum ditemukan kandidat. Periksa laporan dan audit query.')
    print('\nCatatan: kandidat harus diperiksa secara semantik, bukan label final.')
    return examples

SAMPLE_TWEETS = preview_policy_samples()


lombok_2018 | contoh kandidat otomatis: 5
1. [Mon Aug 06 16:23:40 +0000 2018] @LawanPoLitikJKW Kalo pemerintah punya sense of crisis, mustinya langsung menetapkan gempa Lombok sbg bencana nasional...
2. [Mon Aug 06 15:49:26 +0000 2018] @Metro_TV Korupsi itu baru bencana nasional kl yg di Lombok itu bencana alam @Fahrihamzah
3. [Mon Aug 06 11:56:38 +0000 2018] #Bencana Nasional #Gempa #Lombok: Kisah Nyata Mengharukan dari Seorang Korban

#WargaSERUJI : Rory Anas | Artikel Warga #NTB #Tsunami
https://t.co/Oi82KYCka8
4. [Mon Aug 06 11:20:11 +0000 2018] Gempa Lombok harus ditetapkan status “Bencana Nasional”, agar penanganan lebih cepat dan intensif. Kasus dua masjid rubuh di Lombok Utara, dg puluhan jamaah diduga berada di bawah reruntuhan, pertanda penanganan masih tergolong lambat...
5. [Mon Aug 06 11:08:02 +0000 2018] BNPB: Tak Perlu Status Bencana Nasional dan Bantuan Asing Untuk Lombok https://t.co/nHIlnljKqw

palu_2018 | contoh kandidat otomatis: 5
1. [Sat Sep 29 16:06:51 +0000 201

## Interpretasi hasil
- `BASIC_CHECKS_PASS` = hanya lolos pemeriksaan otomatis dasar, bukan jaminan relevansi semantik/kelengkapan.
- `NO_POLICY_CANDIDATE` = jangan langsung menganggap tidak ada opini; evaluasi operator query, rentang, dan aktor.
- `POSSIBLE_TRUNCATION` = potensi kehilangan hasil. Setelah uji pilot, tinjau pembagian interval dan kredit.
- `MOCK_PRESENT`, `NO_REAL_TWEET`, `OUT_OF_WINDOW` = kualitas perlu ditinjau sebelum Tahap 02.
- `FAILED` = panggilan gagal dan dapat dicoba pada eksekusi berikutnya, tanpa mengulang task yang telah disimpan.
- Dataset tahap 01 **bukan** training set LSTM. Tahap 02 melakukan cleaning dan Tahap 03 labeling.

Untuk FULL: ubah `RUN_MODE = 'full'`, pastikan pilot telah diaudit, lalu aktifkan
`EXECUTE_SCRAPING = True` dan tetapkan `MAX_NEW_RUNS` sesuai batas waktu/kredit.
Jalankan ulang dari awal supaya fungsi memakai konfigurasi yang baru.
Jangan commit folder `outputs/` atau token ke repository publik.